In [ ]:
from RMtools_3D import do_RMsynth_3D
from RMtools_3D import do_RMclean_3D
from RMtools_3D import do_fitIcube
from RMtools_1D import do_RMsynth_1D
from RMtools_1D import do_RMclean_1D
from RMutils import util_RM
import importlib
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import WCS

In [ ]:
dir_in = '/srv/data/dva/dragons_hoard/Data_cubes_Sept_2025_BB/convolved_masked_cubes/350to1000/conv_plat_car/'

hdu    = fits.open(dir_in + 'q_350to1000MHz_convolved_3.6deg_masked_gal_K.fits')
q_data = hdu[0].data
hdu    = fits.open(dir_in + 'u_350to1000MHz_convolved_3.6deg_masked_gal_K.fits')
u_data = hdu[0].data
hdu    = fits.open(dir_in + 'i_350to1000MHz_convolved_3.6deg_masked_gal_Kcorr.fits')
i_data = hdu[0].data
hdu    = fits.open(dir_in + 'dQU_combine_gal_K.fits')
qu_err = hdu[0].data

freqs = np.genfromtxt('/srv/data/dva/dragons_hoard/Data_cubes_Jun_2025_BB/ListOfFrequencies_Hz.txt')
print(len(freqs))
print(q_data.shape)
print(u_data.shape)
print(i_data.shape)
print(qu_err.shape)


In [ ]:
lat = WCS(hdu[0].header).all_pix2world(0,range(hdu[0].data.shape[1]),0,0)[1]
lon = WCS(hdu[0].header).all_pix2world(range(hdu[0].data.shape[2]),0,0,0)[0]

In [ ]:
dir_in = '/srv/data/dva/dragons_hoard/Data_cubes_Sept_2025_BB/convolved_masked_cubes/350to1000/conv_plat_car/rmsynth_coarse/'
hdu = fits.open(dir_in + 'five_sigma.fits')
thresh_new = hdu[0].data


In [ ]:
def compare_super_res(lpick, bpick):

    lidx = abs(lpick - lon).argmin()
    bidx = abs(bpick - lat).argmin()
    print(lat[bidx])
    print(lon[lidx])
    
    mdict, adict = do_RMsynth_1D.run_rmsynth([freqs, i_data[:,bidx,lidx], q_data[:,bidx,lidx], u_data[:,bidx,lidx], 
                                                      np.ones(len(freqs)), qu_err[:,bidx,lidx], qu_err[:,bidx,lidx]],
                                                      phiMax_radm2=150, dPhi_radm2=0.2, 
                                                      polyOrd=5, weightType='variance',fitRMSF=True, verbose=False)
    
    mdict2, adict2 = do_RMsynth_1D.run_rmsynth([freqs, i_data[:,bidx,lidx], q_data[:,bidx,lidx], u_data[:,bidx,lidx], 
                                                      np.ones(len(freqs)), qu_err[:,bidx,lidx], qu_err[:,bidx,lidx]],
                                                      phiMax_radm2=150, dPhi_radm2=0.2, 
                                                      polyOrd=5, weightType='variance',fitRMSF=True, verbose=False, super_resolution=True)
    
    mdict_cl, adict_cl = do_RMclean_1D.run_rmclean(mdict, adict, thresh_new[bidx,lidx], maxIter=1000, gain=0.1, 
                                                    showPlots=False, prefixOut='', verbose=False)
    
    mdict_cl2, adict_cl2 = do_RMclean_1D.run_rmclean(mdict2, adict2, thresh_new[bidx,lidx], maxIter=1000, gain=0.1, 
                                                    showPlots=False, prefixOut='', verbose=False)
    
    fig, ax = plt.subplots(3,1,figsize=(12,10))
    
    ax[0].plot(adict['phiArr_radm2'],    abs(adict['dirtyFDF']),    linewidth=2)
    ax[0].plot(adict2['phiArr_radm2'],    abs(adict2['dirtyFDF']),    linewidth=2)
    ax[0].axhline(y=thresh_new[bidx,lidx], linewidth=2, linestyle='dashed', color='grey')
    
    ax[1].plot(adict_cl['phiArr_radm2'], abs(adict_cl['cleanFDF']), linewidth=2)
    ax[1].plot(adict_cl2['phiArr_radm2'], abs(adict_cl2['cleanFDF']), linewidth=2)
    ax[1].plot(adict_cl['phiArr_radm2'], abs(adict_cl['ccArr']), linewidth=2,color='C0')
    ax[1].plot(adict_cl2['phiArr_radm2'], abs(adict_cl2['ccArr']), linewidth=2,color='C1')
    ax[1].axhline(y=thresh_new[bidx,lidx], linewidth=2, linestyle='dashed', color='grey')
    
    ax[2].plot(adict['phi2Arr_radm2'],    abs(adict['RMSFArr']),    linewidth=2)
    ax[2].plot(adict2['phi2Arr_radm2'],    adict2['RMSFArr'].imag,    linewidth=2)
    
    for i in range(0,3):
        ax[i].set_xlim(-100,100)

    #print(adict_cl.keys())

    return
    

In [ ]:
#compare_super_res(20.0, 45.0)
#compare_super_res(150.0, 30.0)
#compare_super_res(137.0, 7.0)
compare_super_res(150.5, 36.4)